<a href="https://colab.research.google.com/github/ronniedebojit2002-netizen/Data-Intelligence-System/blob/main/Part3_Advanced_Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Applied AI & ML Capstone Project

# Part 3 – Advanced Modeling: Ensembles, Hyperparameter Tuning and ML Pipeline

**Student:** Debojit Lahiri

---

## Objective

The objective of Part 3 is to compare advanced machine learning models, perform hyperparameter tuning, evaluate models using cross-validation, and build a complete reusable machine learning pipeline. The final optimized model is serialized and reloaded for future predictions.

In [3]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import joblib

# preprocessing
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV
)

from sklearn.preprocessing import StandardScaler

from sklearn.pipeline import make_pipeline

from sklearn.impute import SimpleImputer

# Models
from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

from sklearn.linear_model import LogisticRegression

# Metrics
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score
)

In [4]:
housing_df = pd.read_csv("cleaned_data.csv")

housing_df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [5]:
# Regression target
y_reg = housing_df["SalePrice"]

# Classification target
y_clf = (
    housing_df["SalePrice"] >
    housing_df["SalePrice"].median()
).astype(int)

# Features
X = housing_df.drop(
    columns=["SalePrice"]
)

In [6]:
quality_map = {
    "Po":1,
    "Fa":2,
    "TA":3,
    "Gd":4,
    "Ex":5
}

ordinal_features = [
    "ExterQual",
    "ExterCond",
    "KitchenQual",
    "HeatingQC"
]

X_encoded = X.copy()

for col in ordinal_features:
    if col in X_encoded.columns:
        X_encoded[col] = X_encoded[col].map(quality_map)

remaining = X_encoded.select_dtypes(
    include=["object","category"]
).columns

X_encoded = pd.get_dummies(
    X_encoded,
    columns=remaining,
    drop_first=True
)

print(X_encoded.shape)

(1460, 235)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y_clf,
    test_size=0.20,
    random_state=42
)

In [8]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

print("Scaling completed.")

Scaling completed.


## 1. Decision Tree Baseline

A baseline Decision Tree classifier is trained using the default parameters (`max_depth=None`). This allows the tree to grow without restriction and serves as a reference model for later comparison.

In [9]:
from sklearn.tree import DecisionTreeClassifier

tree_default = DecisionTreeClassifier(
    random_state=42
)

tree_default.fit(
    X_train_scaled,
    y_train
)

print("Baseline Decision Tree trained successfully.")

Baseline Decision Tree trained successfully.


In [10]:
train_pred = tree_default.predict(X_train_scaled)
test_pred = tree_default.predict(X_test_scaled)

train_accuracy = accuracy_score(
    y_train,
    train_pred
)

test_accuracy = accuracy_score(
    y_test,
    test_pred
)

print("Baseline Decision Tree")

print("-"*35)

print(f"Training Accuracy : {train_accuracy:.4f}")
print(f"Testing Accuracy  : {test_accuracy:.4f}")

Baseline Decision Tree
-----------------------------------
Training Accuracy : 1.0000
Testing Accuracy  : 0.8973


## 2. Controlled Decision Tree

A second Decision Tree classifier is trained with constraints on tree depth and minimum samples required for splitting. These restrictions reduce overfitting and improve generalization.

In [11]:
tree_controlled = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20,
    random_state=42
)

tree_controlled.fit(
    X_train_scaled,
    y_train
)

print("Controlled Decision Tree trained successfully.")

Controlled Decision Tree trained successfully.


In [12]:
train_pred_controlled = tree_controlled.predict(
    X_train_scaled
)

test_pred_controlled = tree_controlled.predict(
    X_test_scaled
)

train_accuracy_controlled = accuracy_score(
    y_train,
    train_pred_controlled
)

test_accuracy_controlled = accuracy_score(
    y_test,
    test_pred_controlled
)

print("Controlled Decision Tree")

print("-"*35)

print(f"Training Accuracy : {train_accuracy_controlled:.4f}")
print(f"Testing Accuracy  : {test_accuracy_controlled:.4f}")

Controlled Decision Tree
-----------------------------------
Training Accuracy : 0.9272
Testing Accuracy  : 0.8767


In [13]:
tree_comparison = pd.DataFrame({

    "Model":[
        "Default Tree",
        "Controlled Tree"
    ],

    "Training Accuracy":[
        train_accuracy,
        train_accuracy_controlled
    ],

    "Testing Accuracy":[
        test_accuracy,
        test_accuracy_controlled
    ]

})

tree_comparison

,Model,Training Accuracy,Testing Accuracy
0,Default Tree,1.000000,0.897260
1,Controlled Tree,0.927226,0.876712


## 3. Gini vs Entropy

Two Decision Tree models are trained using different splitting criteria to compare their classification performance.

In [14]:
gini_tree = DecisionTreeClassifier(
    criterion="gini",
    max_depth=5,
    random_state=42
)

entropy_tree = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=5,
    random_state=42
)

gini_tree.fit(
    X_train_scaled,
    y_train
)

entropy_tree.fit(
    X_train_scaled,
    y_train
)

gini_accuracy = accuracy_score(
    y_test,
    gini_tree.predict(X_test_scaled)
)

entropy_accuracy = accuracy_score(
    y_test,
    entropy_tree.predict(X_test_scaled)
)

comparison = pd.DataFrame({

    "Criterion":[
        "Gini",
        "Entropy"
    ],

    "Test Accuracy":[
        gini_accuracy,
        entropy_accuracy
    ]

})

comparison

,Criterion,Test Accuracy
0,Gini,0.893836
1,Entropy,0.900685


## 4. Random Forest

A Random Forest classifier is trained using an ensemble of decision trees. Bootstrap sampling and random feature selection are used to reduce variance and improve generalization performance.

In [15]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf_model.fit(
    X_train_scaled,
    y_train
)

print("Random Forest trained successfully.")

Random Forest trained successfully.


In [16]:
rf_train_pred = rf_model.predict(X_train_scaled)

rf_test_pred = rf_model.predict(X_test_scaled)

rf_train_accuracy = accuracy_score(
    y_train,
    rf_train_pred
)

rf_test_accuracy = accuracy_score(
    y_test,
    rf_test_pred
)

rf_auc = roc_auc_score(
    y_test,
    rf_model.predict_proba(X_test_scaled)[:,1]
)

print("Random Forest Results")

print("-"*35)

print(f"Training Accuracy : {rf_train_accuracy:.4f}")

print(f"Testing Accuracy  : {rf_test_accuracy:.4f}")

print(f"ROC-AUC           : {rf_auc:.4f}")

Random Forest Results
-----------------------------------
Training Accuracy : 0.9957
Testing Accuracy  : 0.9452
ROC-AUC           : 0.9844


In [17]:
feature_importance = pd.DataFrame({

    "Feature": X_encoded.columns,

    "Importance": rf_model.feature_importances_

})

feature_importance = feature_importance.sort_values(

    by="Importance",

    ascending=False

)

feature_importance.head(10)

,Feature,Importance
19,GrLivArea,0.083904
6,YearBuilt,0.060458
4,OverallQual,0.059282
22,FullBath,0.051729
14,TotalBsmtSF,0.037707
31,GarageArea,0.036981
26,KitchenQual,0.036075
9,ExterQual,0.035474
30,GarageCars,0.033401
16,1stFlrSF,0.025599


In [18]:
print("Top Five Important Features")

feature_importance.head(5)

Top Five Important Features


,Feature,Importance
19,GrLivArea,0.083904
6,YearBuilt,0.060458
4,OverallQual,0.059282
22,FullBath,0.051729
14,TotalBsmtSF,0.037707


## 5. Gradient Boosting

Gradient Boosting sequentially trains weak learners where each tree attempts to correct the mistakes made by the previous trees.

In [19]:
gb_model = GradientBoostingClassifier(

    n_estimators=100,

    learning_rate=0.1,

    max_depth=3,

    random_state=42

)

gb_model.fit(

    X_train_scaled,

    y_train

)

gb_train = accuracy_score(

    y_train,

    gb_model.predict(X_train_scaled)

)

gb_test = accuracy_score(

    y_test,

    gb_model.predict(X_test_scaled)

)

gb_auc = roc_auc_score(

    y_test,

    gb_model.predict_proba(X_test_scaled)[:,1]

)

print("Gradient Boosting")

print("-"*35)

print(f"Training Accuracy : {gb_train:.4f}")

print(f"Testing Accuracy  : {gb_test:.4f}")

print(f"ROC-AUC           : {gb_auc:.4f}")

Gradient Boosting
-----------------------------------
Training Accuracy : 0.9889
Testing Accuracy  : 0.9452
ROC-AUC           : 0.9857


## 6. Feature Ablation Study

The five least important features identified by the Random Forest are removed, and a second Random Forest model is trained using identical hyperparameters. This experiment evaluates whether removing weak features affects predictive performance.

In [20]:
# Five least important features
lowest_features = feature_importance.tail(5)

print("Five Lowest Importance Features")

lowest_features

Five Lowest Importance Features


,Feature,Importance
185,Electrical_Mix,0.0
206,GarageQual_Po,0.0
223,SaleType_Con,0.0
219,MiscFeature_Othr,0.0
221,MiscFeature_TenC,0.0


In [21]:
lowest_feature_names = lowest_features["Feature"].tolist()

X_train_reduced = X_train.drop(columns=lowest_feature_names)

X_test_reduced = X_test.drop(columns=lowest_feature_names)

# Scale reduced features
scaler_reduced = StandardScaler()

X_train_reduced_scaled = scaler_reduced.fit_transform(X_train_reduced)

X_test_reduced_scaled = scaler_reduced.transform(X_test_reduced)

In [22]:
rf_reduced = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf_reduced.fit(
    X_train_reduced_scaled,
    y_train
)

auc_reduced = roc_auc_score(
    y_test,
    rf_reduced.predict_proba(X_test_reduced_scaled)[:,1]
)

print("Full Model AUC :", rf_auc)

print("Reduced Model AUC :", auc_reduced)

Full Model AUC : 0.9844483428950737
Reduced Model AUC : 0.9839742070077284


## 7. Cross-Validated Model Comparison

Five-fold stratified cross-validation is performed to compare Logistic Regression, Controlled Decision Tree, Random Forest, and Gradient Boosting using ROC-AUC.

In [23]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

models = {

    "Logistic Regression":
        LogisticRegression(max_iter=1000),

    "Controlled Decision Tree":
        DecisionTreeClassifier(
            max_depth=5,
            min_samples_split=20,
            random_state=42
        ),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            random_state=42
        ),

    "Gradient Boosting":
        GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=42
        )
}

cv_results = []

for name, model in models.items():

    scores = cross_val_score(

        model,

        X_encoded,

        y_clf,

        cv=cv,

        scoring="roc_auc"

    )

    cv_results.append([

        name,

        scores.mean(),

        scores.std()

    ])

cv_table = pd.DataFrame(

    cv_results,

    columns=[

        "Model",

        "Mean AUC",

        "Std AUC"

    ]

)

cv_table

,Model,Mean AUC,Std AUC
0,Logistic Regression,0.965828,0.011926
1,Controlled Decision Tree,0.920997,0.012702
2,Random Forest,0.980259,0.004404
3,Gradient Boosting,0.978035,0.004454


## 8. Hyperparameter Tuning using GridSearchCV

A machine learning pipeline is created using a median imputer, StandardScaler, and RandomForestClassifier. GridSearchCV is then used to search for the best hyperparameter combination using five-fold stratified cross-validation and ROC-AUC as the evaluation metric.

In [24]:
pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

pipeline

Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='median')),
                ('standardscaler', StandardScaler()),
                ('randomforestclassifier',
                 RandomForestClassifier(random_state=42))])

In [25]:
param_grid = {

    "randomforestclassifier__n_estimators":[50,100,200],

    "randomforestclassifier__max_depth":[5,10,None],

    "randomforestclassifier__min_samples_leaf":[1,5]

}

In [26]:
grid = GridSearchCV(

    pipeline,

    param_grid,

    cv=StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    ),

    scoring="roc_auc",

    n_jobs=-1

)

grid.fit(
    X_train,
    y_train
)

print("Best Parameters")

print(grid.best_params_)

print("\nBest CV Score")

print(grid.best_score_)

Best Parameters
{'randomforestclassifier__max_depth': 10, 'randomforestclassifier__min_samples_leaf': 1, 'randomforestclassifier__n_estimators': 200}

Best CV Score
0.9782964711437587


## 9. Manual Learning Curve

The best pipeline obtained from GridSearchCV is trained using progressively larger fractions of the training dataset to evaluate how model performance changes as more training data becomes available.

In [27]:
fractions = [0.2,0.4,0.6,0.8,1.0]

learning_results=[]

best_pipeline = grid.best_estimator_

for fraction in fractions:

    size = int(fraction * len(X_train))

    X_sub = X_train.iloc[:size]

    y_sub = y_train.iloc[:size]

    best_pipeline.fit(
        X_sub,
        y_sub
    )

    train_auc = roc_auc_score(
        y_sub,
        best_pipeline.predict_proba(X_sub)[:,1]
    )

    test_auc = roc_auc_score(
        y_test,
        best_pipeline.predict_proba(X_test)[:,1]
    )

    learning_results.append([
        fraction,
        train_auc,
        test_auc
    ])

learning_curve = pd.DataFrame(

    learning_results,

    columns=[
        "Training Fraction",
        "Training AUC",
        "Test AUC"
    ]

)

learning_curve

,Training Fraction,Training AUC,Test AUC
0,0.2,1.000000,0.979778
1,0.4,1.000000,0.982457
2,0.6,0.999992,0.984354
3,0.8,0.999986,0.984164
4,1.0,0.999941,0.984448


In [30]:
joblib.dump(
    best_pipeline,
    "best_model.pkl"
)


['best_model.pkl']

In [31]:
loaded_model = joblib.load(
    "best_model.pkl"
)

sample_rows = X_test.iloc[:2]

predictions = loaded_model.predict(
    sample_rows
)

print("Predictions")

print(predictions)

Predictions
[0 1]
